Model for bank churn prediction :

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

df = pd.read_csv('Bank Customer Churn Prediction.csv')
df_clean = df.drop('customer_id', axis=1)
df_encoded = pd.get_dummies(df_clean, columns=['country', 'gender'], drop_first=True)

X = df_encoded.drop('churn', axis=1)
y = df_encoded['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train_res, y_train_res)

y_probs = model.predict_proba(X_test_scaled)[:, 1]

best_threshold = 0.47
preds = (y_probs >= best_threshold).astype(int)


print(f"Overall Accuracy: {accuracy_score(y_test, preds):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))

Overall Accuracy: 0.8320

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      1593
           1       0.58      0.63      0.60       407

    accuracy                           0.83      2000
   macro avg       0.74      0.76      0.75      2000
weighted avg       0.84      0.83      0.83      2000



testing with  custom inputs :::

In [25]:

random_customer = pd.DataFrame([{
    'credit_score': 650,
    'age': 42,
    'tenure': 3,
    'balance': 50000.00,
    'products_number': 2,
    'credit_card': 1,
    'active_member': 0,
    'estimated_salary': 75000.00,
    'country_Germany': 1,
    'country_Spain': 0,
    'gender_Male': 1
}])

customer_scaled = scaler.transform(random_customer)


prob = model.predict_proba(customer_scaled)[:, 1][0]
prediction = 1 if prob >= best_threshold else 0

print(f"--- Test Input Result ---")
print(f"Churn Probability: {prob}")
print(f"Prediction (Threshold {best_threshold}): {'CHURN' if prediction == 1 else 'STAY'}")

--- Test Input Result ---
Churn Probability: 0.25
Prediction (Threshold 0.47): STAY
